In [3]:
import pandas as pd
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from IPython.display import display

# --- 1. SETUP BASE MODEL ---
# Using the standard pre-trained GPT-2 (general text)
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def generate_base_text(prompt, temp=0.7, k=50, p=0.9):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    output_ids = model.generate(
        input_ids,
        max_length=60,
        temperature=temp,
        top_k=k,
        top_p=p,
        do_sample=True,
        no_repeat_ngram_size=2,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# --- 2. PARAMETER INFLUENCE ANALYSIS ---
test_prompt = "The Song of Hiawatha is based on"

# Temperature Analysis (Creativity)
temperature_vals = [0.3, 0.8, 1.5]
results1 = [generate_base_text(test_prompt, temp=t) for t in temperature_vals]

# Top-K Analysis (Vocabulary Filtering)
top_k_vals = [5, 40, 100]
results2 = [generate_base_text(test_prompt, k=tk) for tk in top_k_vals]

# Top-P Analysis (Nucleus Sampling)
top_p_vals = [0.2, 0.8, 0.99]
results3 = [generate_base_text(test_prompt, p=tp) for tp in top_p_vals]

# --- 3. TOKENIZER INFLUENCE ANALYSIS ---
words_to_test = ["Hiawatha", "Ojibway", "forest-fire", "AI-generated"]
token_breakdown = {word: tokenizer.tokenize(word) for word in words_to_test}

# --- 4. FORMATTING RESULTS ---
df_temp = pd.DataFrame({"Valoare": temperature_vals, "Text Generat": results1})
df_top_k = pd.DataFrame({"Valoare": top_k_vals, "Text Generat": results2})
df_top_p = pd.DataFrame({"Valoare": top_p_vals, "Text Generat": results3})

# Display tables
print("### 1. Temperature Influence (Base Model)")
display(df_temp.style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}).format({"Valoare": "{:.1f}"}))

print("\n### 2. Top-K Influence (Base Model)")
display(df_top_k.style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))

print("\n### 3. Top-P Influence (Base Model)")
display(df_top_p.style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}).format({"Valoare": "{:.1f}"}))

# Display Tokenizer Analysis
print("\n### 4. Tokenizer Analysis (BPE Breakdown)")
for word, tokens in token_breakdown.items():
    print(f"Word: {word:15} | Tokens: {tokens}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

### 1. Temperature Influence (Base Model)


,Valoare,Text Generat
0,0.3,"The Song of Hiawatha is based on the story of the Song, which is a story about a young woman who has been kidnapped by a group of bandits. The song is about the journey of a girl who is kidnapped from her family by the bandits, and the fate of her parents."
1,0.8,"The Song of Hiawatha is based on the works of an old Chinese poet and translator. It is an ancient Chinese poem, meaning ""Song of the Song"". The song is written in a traditional Chinese style, including the two characters, the sound of which can be heard through the lips of"
2,1.5,The Song of Hiawatha is based on that poem but as far as the Song does exist there are only limited variations due to the various references found on both the web of different authors. The Song that appears in the Dragonfire novelis based upon that poet of whom Nynaeve states



### 2. Top-K Influence (Base Model)


,Valoare,Text Generat
0,5,"The Song of Hiawatha is based on the Song from the Bible. It is a song of the people of God. The Song is about the life of a man and a woman. The song was written in the year 2000 and was composed by the poet, S. Srin"
1,40,"The Song of Hiawatha is based on a poem by the Chinese poet Hui-Ting, which is in the form of a story about a prince, and the story is narrated by Hao-Chuan, the prince's son, who is a famous poet and has been a"
2,100,"The Song of Hiawatha is based on the epic poem ""The Book of Songs,"" by the same author. The book is a collection of poetry and songs that was written by a woman named Hyawata (Jame) of the Iliad. The book was published"



### 3. Top-P Influence (Base Model)


,Valoare,Text Generat
0,0.2,"The Song of Hiawatha is based on the story of a young girl who is kidnapped by a group of bandits. The girl is forced to join the band of thieves and is eventually rescued by the group. The song is a parody of the song ""The Little Mermaid"" by Disney"
1,0.8,"The Song of Hiawatha is based on the same poem as the poem of the Song in the West, which was written by the famous poet, Thomas S. Eliot. The Song is a tale of a man and a woman who are in love, and the man, who is the first"
2,1.0,"The Song of Hiawatha is based on the Song Of Humbug, the poem of a monk who was sent by the Emperor to the North. Hiyun was the son of the King of Hyrule and the leader of his tribe. He was also the brother of Tifa."



### 4. Tokenizer Analysis (BPE Breakdown)
Word: Hiawatha        | Tokens: ['H', 'ia', 'w', 'atha']
Word: Ojibway         | Tokens: ['O', 'j', 'ib', 'way']
Word: forest-fire     | Tokens: ['forest', '-', 'fire']
Word: AI-generated    | Tokens: ['AI', '-', 'generated']


In [2]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    GPT2Tokenizer, 
    GPT2LMHeadModel, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForLanguageModeling
)
from IPython.display import display

# --- 1. DATA PREPARATION ---
df_raw = pd.read_parquet('data/poetries.parquet')

# Grouping lines by 'gutenberg_id'
df_grouped = df_raw.groupby('gutenberg_id')['line'].apply(lambda x: "\n".join(x)).reset_index()
dataset = Dataset.from_pandas(df_grouped[['line']])

# Initialize Tokenizer and Model
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["line"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["line"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# --- 2. THE OPTIMIZATION PIPELINE (Fine-Tuning) ---
training_args = TrainingArguments(
    output_dir="./poetry_model_v2",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    save_steps=1000,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
)

trainer.train()

# --- 3. PARAMETER ANALYSIS ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def test_generation(prompt, temp=0.7, p=0.9):
    ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    out = model.generate(
        ids,
        max_length=60,
        temperature=temp,
        top_p=p,
        do_sample=True,
        no_repeat_ngram_size=2,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

# Starting verse for the experiment
start_verse = "The song of the forest began"

# Analyze Temperature influence
temps = [0.2, 0.7, 1.2]
results_temp = [test_generation(start_verse, temp=t) for t in temps]
df_temp_analysis = pd.DataFrame({"Temperature": temps, "Generated Text": results_temp})

# Analyze Top-p influence
ps = [0.5, 0.9, 0.99]
results_p = [test_generation(start_verse, p=pv) for pv in ps]
df_p_analysis = pd.DataFrame({"Top-P": ps, "Generated Text": results_p})

# --- 4. DISPLAY RESULTS ---
print("### Influence of Temperature")
display(df_temp_analysis.style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))

print("\n### Influence of Top-P (Nucleus Sampling)")
display(df_p_analysis.style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))

# --- 5. FINAL POEM COMPLETION ---
# Using balanced parameters for the final output
print("\n--- Final Poem Completion ---")
print(test_generation("Deep in the valley where shadows lie", temp=0.8, p=0.92))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Map:   0%|          | 0/1191 [00:00<?, ? examples/s]

C:\Users\paulf\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,4.295448
200,4.213119
300,4.192679
400,3.887545
500,3.864171
600,3.827481
700,3.727625
800,3.686014


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

### Influence of Temperature


,Temperature,Generated Text
0,0.200000,"The song of the forest began When the sun was shining, And the moon was rising, and the stars were rising In the sky. The sun rose, the starry stars rose As the night went by,-- For the first time in the year I saw the"
1,0.700000,"The song of the forest began, And the wind-cloud grew dark, and the sea-floor Was cold and cloud-grey; Its waves were like a phantom,-- I could hear A sea of my own song, to hear it The music of a forest."
2,1.200000,"The song of the forest began to tremble, And Whispering: Sorrow and fear, as you walk, With fear on your breast: But all this comes to you, I will let go."" ""'Tis I my job,' she'd say. A song"



### Influence of Top-P (Nucleus Sampling)


,Top-P,Generated Text
0,0.500000,"The song of the forest began to grow With the wind that blew The wind of spring And the moon that sang In the night-time That came down from the sky To the land of her birth Was the song Of the wild, the green, and the white"
1,0.900000,"The song of the forest began, And the tree was not long in view But it was growing, and was waiting For the morning to bloom. It was the dawn of a new year,-- The sun shone through the sky, bright With a smile of joy and hope"
2,0.990000,"The song of the forest began With a soft, sweet sigh. I heard a loud and strange murmur Of birds. I heard the birds singing In the night. A little nest of young trees Echety'd in the shade And all the while the little tree"



--- Final Poem Completion ---
Deep in the valley where shadows lie,
I sing the songs of the birds, and the dogs, by whom
That sweetly they call home, when the stars are burning,--
A song of a song that I cannot sing, but know, that
It is sung of my beloved


c.1 Diferențe de calitate

    - LLM General: Bun la gramatică, dar scrie proză sau fapte. Ignoră structura poetică (rime, strofe).

    - LLM Specializat: Învață "formatul" de poezie (versuri scurte, ritm), chiar dacă devine mai puțin informat general.

c.2 Prompt în Engleză

    - Funcționează perfect. GPT-2 este nativ în engleză, deci tokenizarea și logica sunt optime. Preia rapid stilul autorilor din corpus (ex: Longfellow).

c.3 Prompt în Română

    - Calitate slabă. Tokenizatorul GPT-2 nu recunoaște cuvintele românești și le sparge în litere individuale. Rezultatul e adesea un amestec de silabe fără sens.

c.4 Prompt Română + Antrenament Engleză

    - Conflict lingvistic. Modelul va încerca să răspundă în engleză sau va inventa cuvinte hibride ("hallucinations"). Nu poate genera poezie coerentă în română dacă a văzut doar engleză.

c.5 Personalizare pentru "Pastel"

    - Date: Filtrează corpusul după cuvinte cheie (natură, soare, flori).

    - Fine-tuning: Antrenează modelul doar pe acest subset "verde".

    - Parametri:

        - Temperature (0.7): Pentru descrieri variate, dar logice.

        - Top-P (0.9): Pentru a folosi adjective plastice și rare.

        - Prompt: Începe cu un vers descriptiv (ex: "The golden field whispers...").

In [6]:
prompt_ro = "Soarele apune printre ramuri"

print("### Prompt Romana:")
print(prompt_ro)
print("\n### Rezultat LLM normal:")
print(generate_base_text(prompt_ro))
print("\n### Rezultat LLM adaptat:")
print(test_generation(prompt_ro))

### Prompt Romana:
Soarele apune printre ramuri

### Rezultat LLM normal:
Soarele apune printre ramuri e ai della vidale, et in vita una dei di nostra, che ils se vinci possa, il di sua suo, per se sui, la vista uno a vie

### Rezultat LLM adaptat:
Soarele apune printre ramuri (I'm talking about the front of the book.)

This is a book about what it means to be a man, a woman, and a human being.

